In [1]:
import os
import sys
sys.path.append("../")
from qdrant_client import QdrantClient
import numpy as np
import polars as pl
import subprocess
from typing import Any, List, Tuple, Iterable, Dict
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)

def load_env(
    secret_names: Iterable[str] = ("QDRANT_API", "QDRANT_ENDPOINT"),
) -> Dict[str, str]:
    """
    Doppler CLI에서 시크릿을 읽어와 현재 프로세스 os.environ 에 주입합니다.
    파일(.env)을 생성하지 않습니다.

    Args:
        secret_names: 가져올 Doppler 시크릿 이름 목록
        project: doppler -p 옵션 (선택)
        config: doppler -c 옵션 (선택)

    Returns:
        {'<name>': 'SET', ...} 형태의 간단한 결과 요약
    """

    base_cmd = ["doppler", "secrets", "get"]

    for name in secret_names:
        try:
            # doppler secrets get <name> --plain
            cmd = base_cmd + [name, "--plain"]
            value = subprocess.check_output(cmd, text=True).strip()
            if not value:
                raise RuntimeError(f"시크릿 '{name}' 값이 비어 있습니다.")
            os.environ[name] = value
        except subprocess.CalledProcessError as e:
            raise RuntimeError(f"Doppler에서 시크릿 '{name}' 조회 실패: {e}") from e

ITEM_COLLECTION = "bite-vectordb"
CAT_COLLECTION  = "category-profiles"
CATEGORY = [i for i in range(1, 14)]
MIN_POINTS = int(os.getenv("MIN_POINTS_PER_CATEGORY", "5"))

load_env()

client = QdrantClient(
    url=os.getenv("QDRANT_ENDPOINT"),
    api_key=os.getenv("QDRANT_API")
)

In [11]:
# ----- 유틸 -----
def l2_normalize(v: np.ndarray) -> np.ndarray:
    if v.ndim == 1:
        n = np.linalg.norm(v)
        return (v / n).astype(np.float32) if n > 0 else v.astype(np.float32)
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n = np.where(n == 0, 1.0, n)
    return (v / n).astype(np.float32)

def get_dim_and_metric(client: QdrantClient, collection: str) -> Tuple[int, str]:
    info = client.get_collection(collection)
    dim = info.config.params.vectors.size
    dist = info.config.params.vectors.distance.value
    return dim, dist


# ----- 카테고리 수집/벡터 수집 -----
def fetch_vectors_by_category(client, category_value: int) -> np.ndarray:
    filt = Filter(must=[FieldCondition(key="category", match=MatchValue(value=category_value))])
    vecs = []
    next_offset = None
    while True:
        points, next_offset = client.scroll(
            collection_name=ITEM_COLLECTION,
            limit=256, 
            with_payload=True, 
            with_vectors=True, 
            offset=next_offset,
            scroll_filter=filt
        )
        for p in points:
            v = p.vector
            if v is not None:
                vecs.append(np.asarray(v, dtype=np.float32))
        if next_offset is None:
            break
    return np.vstack(vecs) if vecs else np.zeros((0,))


def init_collection(client, collection: str, dim: int):
    if not client.collection_exists(collection):
        client.create_collection(
            collection_name=collection,
            vectors_config=VectorParams(
                size=dim,
                distance=Distance.COSINE
            )
        )

# ----- 메인 로직 -----
def build_category_profiles(client, min_points: int = MIN_POINTS):
    dim, _ = get_dim_and_metric(client, ITEM_COLLECTION)      # 아이템 컬렉션 기준으로 맞춤l;
    init_collection(
        client,
        CAT_COLLECTION,
        dim
    )

    created, skipped = 0, 0

    for c in CATEGORY:
        V = fetch_vectors_by_category(client, c)
        if V.shape[0] < min_points:
            skipped += 1
            continue

        # 아이템 벡터가 이미 정규화라면 단순 평균 후 재정규화
        centroid = l2_normalize(V.mean(axis=0))

        client.upsert(
            collection_name=CAT_COLLECTION,
            points=[
                PointStruct(
                    id=c,
                    vector=centroid.tolist(),
                    payload={"num_items": int(V.shape[0])}
                )
            ]
        )
        created += 1

    print(f"[CategoryProfiles] created={created}, skipped={skipped}")

In [12]:
build_category_profiles(
    client,
    min_points=3
)

[CategoryProfiles] created=13, skipped=0
